# 在 Bedrock 上微调 Claude 3 Haiku
在本笔记本中，我们将带你完成在 Amazon Bedrock 上微调 Claude 3 Haiku 的全过程。

## 你需要准备什么
- 一个可访问 Bedrock 的 AWS 账户
- 一个数据集（也可以直接使用这里提供的示例数据集）
- [一个能够访问存放训练数据的 s3 桶的服务角色](https://docs.aws.amazon.com/bedrock/latest/userguide/model-customization-iam-role.html)

## 安装依赖

In [ ]:
%pip install boto3

In [ ]:
import boto3

## 准备数据集
用于 Bedrock 微调的数据集必须是 JSONL 文件（即每行一个 JSON 对象的文件）。

JSONL 文件中的每一行都应是一个具有以下结构的 JSON 对象：

```
{
  "system": "<optional_system_message>",
  "messages": [
    {"role": "user", "content": "user message"},
    {"role": "assistant", "content": "assistant response"},
    ...
  ]
}
```

- `system` 字段是可选的。
- 必须至少包含两条消息。
- 第一条消息必须来自 “user”。
- 最后一条消息必须来自 “assistant”。
- 用户消息与助手消息必须交替出现。
- 不允许出现多余的键。


## 示例数据集——JSON 模式
我们包含了一个示例数据集，用于教会模型对所有问题都以 JSON 作答。该数据集如下所示：

In [ ]:
import json

sample_dataset = []
dataset_path = "datasets/json_mode_dataset.jsonl"
with open(dataset_path) as f:
    for line in f:
        sample_dataset.append(json.loads(line))

print(json.dumps(sample_dataset[0], indent=2))

## 将数据集上传到 S3
用于微调的数据集应存放在 s3 上；在本演示中，我们把示例数据集写入一个由你控制的 s3 桶。

In [ ]:
bucket_name = "YOUR_BUCKET_NAME"
s3_path = "json_mode_dataset.jsonl"

s3 = boto3.client("s3")
s3.upload_file(dataset_path, bucket_name, s3_path)

## 启动 Bedrock 微调任务

数据集准备就绪后，你就可以用 `boto3` 启动微调任务了。首先为任务配置几个参数：

In [ ]:
# Configuration
job_name = "anthropic-finetuning-cookbook-training"
custom_model_name = "anthropic_finetuning_cookbook"
role = "YOUR_AWS_SERVICE_ROLE_ARN"
output_path = f"s3://{bucket_name}/finetuning_example_results/"
base_model_id = (
    "arn:aws:bedrock:us-east-1::foundation-model/anthropic.claude-haiku-4-5-20251001-v1:0:200k"
)

# Hyperparameters
epoch_count = 5
batch_size = 4
learning_rate_multiplier = 1.0

然后我们就可以用 `boto3` 启动任务。

In [ ]:
bedrock = boto3.client(service_name="bedrock")
bedrock_runtime = boto3.client(service_name="bedrock-runtime")

bedrock.create_model_customization_job(
    customizationType="FINE_TUNING",
    jobName=job_name,
    customModelName=custom_model_name,
    roleArn=role,
    baseModelIdentifier=base_model_id,
    hyperParameters={
        "epochCount": f"{epoch_count}",
        "batchSize": f"{batch_size}",
        "learningRateMultiplier": f"{learning_rate_multiplier}",
    },
    trainingDataConfig={"s3Uri": f"s3://{bucket_name}/{s3_path}"},
    outputDataConfig={"s3Uri": output_path},
)

训练期间，你可以用它查看任务状态：

In [ ]:
# Check for the job status
status = bedrock.get_model_customization_job(jobIdentifier=job_name)["status"]

## 使用你微调后的模型！

要使用你微调后的模型，[你需要通过 Amazon Bedrock 中的 Provisioned Throughput（预置吞吐量）托管它](https://docs.aws.amazon.com/bedrock/latest/userguide/model-customization-use.html)。模型在 Provisioned Throughput 上就绪后，即可通过 Bedrock API 调用它。

In [ ]:
provisioned_throughput_arn = "YOUR_PROVISIONED_THROUGHPUT_ARN"

In [ ]:
bedrock = boto3.client("bedrock-runtime", region_name="us-east-1")
body = json.dumps(
    {
        "anthropic_version": "bedrock-2023-05-31",
        "max_tokens": 1000,
        "system": "JSON Mode: Enabled",
        "messages": [
            {
                "role": "user",
                "content": [{"type": "text", "text": "What is a large language model?"}],
            }
        ],
    }
)
response = bedrock_runtime.invoke_model(modelId=provisioned_throughput_arn, body=body)
body = json.loads(response["body"].read().decode("utf-8"))

In [ ]:
print(body["content"][0]["text"])